In [1]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from src.maxcut import MaxCut
from src.simulator import QuimbSimulator
from src.torch_simulator import TorchMPSSimulator
from src.optimizer import COBYLA, SMO
from src.lvqe import LayerVQE
from collections import Counter

In [2]:
SEED=562
N_LAYERS=2
K_PER_LAYER=50
K_FINAL=300

SIMULATOR=SMO

num_nodes = 32
def get_random_graph(N: int, seed: int = SEED, plot=False):
    assert N % 2 == 0
    while True:
        G = nx.random_regular_graph(3, num_nodes, seed=seed)
        if nx.is_connected(G):
            if plot:
                plt.figure(figsize=(8, 6))
                pos = nx.spring_layout(G, seed=seed)
                nx.draw(G, pos, with_labels=True, node_color='lightblue',
                    node_size=400, font_size=10, font_weight='bold')
                plt.title(f"Random regular graph G(k=3, N={G.number_of_nodes()}): {G.number_of_edges()} edges")
                plt.show()
            return G
        else:
            seed += 1
    return None

G = get_random_graph(num_nodes, seed=SEED, plot=False)

np.random.seed(SEED)

In [3]:
problem = MaxCut(G, seed=SEED)
print(f"\nn_qubits: {problem.num_qubits}")
print(f"n_terms: {len(problem.hamiltonian_terms)}\n")

sim = TorchMPSSimulator(device='mps')

best_known = problem.best_known_value

print(f"best known cut value : {best_known}")
print(f"corresponding energy: {problem.cut_to_energy(best_known)}")



n_qubits: 31
n_terms: 48

best known cut value : 44.0
corresponding energy: 20.0


In [4]:
# run L-VQE
lvqe = LayerVQE(
    problem=problem,
    simulator=sim,
    optimizer_class=SIMULATOR,
    n_layers=N_LAYERS,
    k_per_layer=K_PER_LAYER,
    k_final=K_FINAL,
    use_sampling=False,
    n_samples=1000,
    record_loss=True
)

result = lvqe.run()

print(f"\nFinal approximation ratio: {result['final_approx_ratio']:.4f}")

# plot the convergence
plt.figure(figsize=(8, 5))
plt.plot(result['history']['layer'], result['history']['approx_ratio'],
         'o-', linewidth=2, markersize=10)
plt.xlabel('Layer')
plt.ylabel('Approximation ratio')
plt.title('L-VQE convergence')
plt.grid(True, alpha=0.3)
plt.xticks(result['history']['layer'])
plt.show()

Starting L-VQE: 2 layers, 50 iter/layer, 300 final iter
Mode: exact expectation

Layer 0: 



SMO: 100%|█████████████████████| 50/50 [00:07<00:00,  6.53it/s, best_E=+13.0000]
/Users/julesdupont/Desktop/qist/Q3/aqaDelft/Layer-Variational-Quantum-Eigensolver/src/torch_simulator.py:86: UserWarning: The operator 'aten::linalg_svd' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:34.)
  U, S, Vh = torch.linalg.svd(theta_mat, full_matrices=False)


layer 0: energy=+13.0000, approx_ratio=+0.8409

Layer 1 — 50 iterations (before convergence)


SMO:  80%|████████████████▊    | 40/50 [00:16<00:04,  2.36it/s, best_E=+14.0000]


KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(8, 5))

optimizer_loss = result['history']['optimizer_loss']
continuous_loss = np.concatenate(optimizer_loss)
print(len(continuous_loss))
plt.plot(continuous_loss, color='crimson', alpha=0.3, linewidth=1.5, label='Loss')

transition_points = [K_PER_LAYER * layer for layer in range(1,N_LAYERS+1)]

for idx, pt in enumerate(transition_points):
    plt.axvline(x=pt, color='black', linestyle='--', alpha=0.6,
                label='Layer Added' if idx == 0 else "")

plt.xlabel('Total Optimization Iterations')
plt.ylabel('Energy (Loss)')
plt.title(f'Training Loss Evolution Across L-VQE Layers')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
bitstrings = sim.get_most_frequent_assignments(result['final_params'], result['final_ansatz'], problem=problem)

for (assignment, proba) in bitstrings:
    print(f"Assignment {assignment} (proba: {proba:.1f})")
    cut_value = problem.evaluate(assignment)
    print(f"Corresponding cut value: {cut_value}")
    print(f"Corresponding energy: {problem.cut_to_energy(cut_value)}")